In [26]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [12]:
documents = []

for file in files:
  doc = file.parse()
  documents.append(doc)

# how many lessons in the pages
print('Number of lessons:', len(documents))
print('First lesson:', documents[0]['filename'])

Number of lessons: 72
First lesson: 01-agentic-rag/lessons/01-intro.md


In [15]:
from minsearch import Index

# create index from docs
index = Index(
  text_fields=['content'],
  keyword_fields=['filename']
)

index.fit(documents)

# quyery the index
query = 'How does the agentic loop keep calling the model until it stops?'
results = index.search(query)
print('results:', results[0]['filename'])

results: 01-agentic-rag/lessons/14-agentic-loop.md


In [19]:
from dotenv import load_dotenv
from openai import OpenAI
import os
from rag_helper import RAGCustom

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=api_key,
)

rag = RAGCustom(index=index, llm_client=client, model='openrouter/free')
answer = rag.rag('How does the agentic loop keep calling the model until it stops?')
print(answer)

Input tokens: 7361
Output tokens: 453
The loop runs **until the model’s response no longer contains a function‑call**.  

1. **Start a while‑True iteration** – the code prints the iteration number and sets a flag `has_function_calls = False`.  
2. **Call the model** with the current `messages` and the list of tools (e.g., `[search_tool]`).  
3. **Collect everything the model returns** (`response.output`).  
4. **Append those items to the conversation history** (`messages.extend(response.output)`).  
5. **Iterate over each output item**:  
   * If it’s a **function call** (`item.type == "function_call"`),  
     * execute the tool (`make_call(item)`),  
     * add the tool’s output back into `messages`, and  
     * set `has_function_calls = True` so the loop will continue.  
   * If it’s a regular **message** (`item.type == "message"`), just print the answer.  
6. **Check the flag** – after processing all items, the code does  

```python
if has_function_calls == False:
    break   # e

In [21]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print('Number of chunks:', len(chunks))

Number of chunks: 295


In [22]:
# index the chunks
chunk_index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

chunk_index.fit(chunks)

# point RAG at chunk index
rag_chunks = RAGCustom(index=chunk_index, llm_client=client, model='openrouter/free')

answer = rag_chunks.rag('How does the agentic loop keep calling the model until it stops?')
print(answer)

Input tokens: 2394
Output tokens: 387
The agentic loop is implemented as an **infinite `while True` loop** that repeatedly:

1. **Calls the model** (`openai_client.responses.create`) with the current message history.  
2. **Processes the model’s output**:
   * If the output contains a **function call** (`tool_call`), the code executes the call, appends the result back into the message history, and sets `has_function_calls = True` so the loop knows a tool was used.  
   * If the output is just a **plain message**, it is printed/handled as the final answer.  
3. **Checks a flag** (`has_function_calls`).  
   * When the model’s response **does not contain any function calls** (`has_function_calls == False`), the loop **breaks** (`if has_function_calls == False: break`).  
   * This break is the exit condition that stops the loop; the model has produced a final answer without any further tool requests.

Because the loop only terminates when the model’s response lacks a `function_call`, the

In [24]:
# from above output
input_tokens = 7361
input_tokens_with_chunks = 2394
print('Input tokens without chunks:', input_tokens)
print('Input tokens with chunks:', input_tokens_with_chunks)
print('Token savings:', input_tokens - input_tokens_with_chunks)
print('Token savings percentage:', (input_tokens - input_tokens_with_chunks) / input_tokens * 100)

Input tokens without chunks: 7361
Input tokens with chunks: 2394
Token savings: 4967
Token savings percentage: 67.47724493954625


In [25]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat.runners import OpenAIChatCompletionsRunner
from toyaikit.chat import IPythonChatInterface
from typing import List, Dict, Any

# 1. define search with counter built in
search_call_count = 0

def search(query: str) -> List[Dict[str, Any]]:
  """
  Search the course materials for relevant content.

  Args:
      query: The search query string.

  Returns:
      A list of matching document chunks.
  """
  global search_call_count
  search_call_count += 1
  print(f"[search call #{search_call_count}] query: {query}")
  return chunk_index.search(query=query, num_results=5)

# 2. register the tool
tools = Tools()
tools.add_tool(search)

# 3. wrap your existing openrouter client
llm_client = OpenAIChatCompletionsClient(
  model='openrouter/free',
  client=client
)

# 4. instructions
AGENT_INSTRUCTIONS = """
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
"""

# 5. build runner
runner = OpenAIChatCompletionsRunner(
  tools=tools,
  developer_prompt=AGENT_INSTRUCTIONS,
  chat_interface=IPythonChatInterface(),
  llm_client=llm_client
)

# 6. run
result = runner.loop(prompt='How does the agentic loop work, and how is it different from plain RAG?')
print(result.last_message)
print(f"\nTotal search calls: {search_call_count}")

[search call #1] query: agentic loop
[search call #2] query: RAG vs agentic loop difference
The **agentic loop** is a framework for building iterative, stateful interactions where an agent (e.g., a model or software system) repeatedly interacts with an environment (e.g., another AI, a user, or tools), adapting its response to evolving contexts while retaining state. Unlike plain RAG (which typically answers static queries by querying a single model), the agentic loop:  
1. **Enables multi-step workflows**: Supports sequences of tool calls, retries, prompt adjustments, and branching logic over multiple turns.  
2. **Retains context**: Stores session state (e.g., user preferences, partial answers) across iterations, avoiding losing progress.  
3. **Dynamically reacts**: Adjusts based on feedback (e.g., retrying failed searches with alternative models) rather than relying on fixed queries.  
4. **Supports flexibility**: Leverages tools like multi-turn conversations, chat interfaces, or up

/workspaces/llm-zoomcamp-2026-code/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:542: UnknownModelWarning: No pricing data for model 'openrouter/free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(
